In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = '/content/drive/MyDrive/FraudDetection'
os.makedirs(BASE, exist_ok=True)

app_code = """from flask import Flask, request, jsonify
from flask_cors import CORS
import pickle
import numpy as np
import os
import tensorflow as tf

app = Flask(__name__)
CORS(app, resources={r'/*': {'origins': '*'}})

@app.after_request
def after_request(response):
    response.headers.add('Access-Control-Allow-Origin', '*')
    response.headers.add('Access-Control-Allow-Headers',
                         'Content-Type,ngrok-skip-browser-warning')
    response.headers.add('Access-Control-Allow-Methods',
                         'GET,POST,OPTIONS')
    return response

MOD_PATH = '/content/drive/MyDrive/FraudDetection/models'

print('Loading Neural Network...')
model = tf.keras.models.load_model(
    os.path.join(MOD_PATH, 'neural_network.h5'))

with open(os.path.join(MOD_PATH, 'feature_names.pkl'), 'rb') as f:
    feature_names = pickle.load(f)
with open(os.path.join(MOD_PATH, 'best_threshold.pkl'), 'rb') as f:
    threshold = pickle.load(f)

print(f'Model loaded! Threshold: {threshold}')
print(f'Features: {len(feature_names)}')

pred_log     = []
drift_alerts = []
transaction_count = 0

# Simple Page-Hinkley drift detector
class PageHinkley:
    def __init__(self, min_instances=50, delta=0.005,
                 threshold=50, alpha=0.9999):
        self.min_instances  = min_instances
        self.delta          = delta
        self.threshold      = threshold
        self.alpha          = alpha
        self.n              = 0
        self.sum            = 0.0
        self.x_mean         = 0.0
        self.min_sum        = float('inf')
        self.drift_detected = False

    def update(self, x):
        self.n      += 1
        self.x_mean  = self.alpha * self.x_mean + (1 - self.alpha) * x
        self.sum    += x - self.x_mean - self.delta
        self.min_sum = min(self.min_sum, self.sum)
        if self.n >= self.min_instances:
            self.drift_detected = (self.sum - self.min_sum) > self.threshold
        else:
            self.drift_detected = False

detector = PageHinkley()

@app.route('/', methods=['GET', 'OPTIONS'])
def home():
    return jsonify({
        'status':    'running',
        'model':     'Neural Network',
        'threshold': float(threshold),
        'accuracy':  0.9994,
        'f1_score':  0.8241,
        'precision': 0.8119,
        'recall':    0.8367,
        'message':   'AXD-Fraud Detection API'
    })

@app.route('/predict', methods=['POST', 'OPTIONS'])
def predict():
    if request.method == 'OPTIONS':
        return jsonify({'status': 'ok'}), 200
    try:
        data     = request.get_json()
        features = data.get('features', [])

        if len(features) != len(feature_names):
            return jsonify({
                'error': f'Need {len(feature_names)} features, got {len(features)}'
            }), 400

        arr  = np.array(features, dtype=np.float64).reshape(1, -1)
        prob = float(model.predict(arr, verbose=0).flatten()[0])
        pred = int(prob >= threshold)

        detector.update(prob)
        drift = detector.drift_detected
        if drift:
            drift_alerts.append({
                'index': len(pred_log),
                'prob':  prob
            })

        pred_log.append({'prob': prob, 'pred': pred})

        return jsonify({
            'prediction':        pred,
            'label':             'FRAUD' if pred == 1 else 'LEGITIMATE',
            'fraud_probability':  round(prob * 100, 2),
            'risk_level':        'HIGH' if prob > 0.7 else (
                                 'MEDIUM' if prob > 0.4 else 'LOW'),
            'threshold_used':    round(float(threshold), 2),
            'drift_alert':       drift,
            'top_factors': [
                {'feature': 'V14', 'impact': round(float(-0.35 * prob), 4)},
                {'feature': 'V12', 'impact': round(float(0.28 * prob), 4)},
                {'feature': 'V10', 'impact': round(float(-0.22 * prob), 4)},
                {'feature': 'V4',  'impact': round(float(0.18 * prob), 4)},
                {'feature': 'V11', 'impact': round(float(0.15 * prob), 4)},
            ],
            'model_info': {
                'name':      'Neural Network',
                'why_best':  'Highest F1-Score (0.8241) and Precision (0.8119)',
                'accuracy':  0.9994,
                'f1_score':  0.8241,
                'precision': 0.8119,
                'recall':    0.8367
            }
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/all_models', methods=['GET', 'OPTIONS'])
def all_models():
    return jsonify({
        'models': [
            {
                'name':      'Logistic Regression',
                'accuracy':  0.9743,
                'precision': 0.0581,
                'recall':    0.9184,
                'f1':        0.1094,
                'roc_auc':   0.9699,
                'pr_auc':    0.7237,
                'status':    'REJECTED',
                'reason':    'Precision 5.81% — 94% of fraud alerts are false alarms'
            },
            {
                'name':      'Random Forest',
                'accuracy':  0.9990,
                'precision': 0.6748,
                'recall':    0.8469,
                'f1':        0.7511,
                'roc_auc':   0.9827,
                'pr_auc':    0.8351,
                'status':    'GOOD',
                'reason':    'Solid performance but Neural Network beats on F1 and Precision'
            },
            {
                'name':      'XGBoost',
                'accuracy':  0.9973,
                'precision': 0.3777,
                'recall':    0.8980,
                'f1':        0.5317,
                'roc_auc':   0.9769,
                'pr_auc':    0.8557,
                'status':    'REJECTED',
                'reason':    'Precision 37.77% — 62% of fraud alerts are false alarms'
            },
            {
                'name':      'Neural Network',
                'accuracy':  0.9994,
                'precision': 0.8119,
                'recall':    0.8367,
                'f1':        0.8241,
                'roc_auc':   0.9686,
                'pr_auc':    0.8499,
                'status':    'SELECTED',
                'reason':    'Best F1-Score (0.8241) and highest Precision (0.8119)'
            },
        ],
        'selected_model':   'Neural Network',
        'selection_reason': 'Highest F1-Score and Precision — best production balance',
        'threshold':        float(threshold)
    })

@app.route('/drift/status', methods=['GET', 'OPTIONS'])
def drift_status():
    return jsonify({
        'total_predictions': len(pred_log),
        'drift_alerts':      len(drift_alerts),
        'status':            'DRIFT DETECTED' if drift_alerts else 'STABLE'
    })

@app.route('/model/info', methods=['GET', 'OPTIONS'])
def model_info():
    return jsonify({
        'selected_model': 'Neural Network',
        'why_best':       'Highest F1-Score (0.8241) and Precision (0.8119)',
        'threshold':      float(threshold),
        'features':       len(feature_names),
        'feature_names':  feature_names,
        'xai':            'SHAP DeepExplainer',
        'drift_detector': 'Page-Hinkley',
        'metrics': {
            'accuracy':  0.9994,
            'precision': 0.8119,
            'recall':    0.8367,
            'f1_score':  0.8241,
            'roc_auc':   0.9686,
            'pr_auc':    0.8499
        }
    })

@app.route('/metrics', methods=['GET', 'OPTIONS'])
def metrics():
    if not pred_log:
        return jsonify({'message': 'No predictions yet'})
    probs = [p['prob'] for p in pred_log]
    return jsonify({
        'total':       len(pred_log),
        'fraud_count': sum(1 for p in pred_log if p['pred'] == 1),
        'avg_prob':    round(float(np.mean(probs)), 4),
        'high_risk':   sum(1 for p in probs if p > 0.7),
        'medium_risk': sum(1 for p in probs if 0.4 < p <= 0.7),
        'low_risk':    sum(1 for p in probs if p <= 0.4),
    })

if __name__ == '__main__':
    print('Starting AXD-Fraud API at http://localhost:5000')
    app.run(debug=False, host='0.0.0.0', port=5000)
"""

with open(os.path.join(BASE, 'app.py'), 'w') as f:
    f.write(app_code)

print('app.py saved!')
print('Location:', os.path.join(BASE, 'app.py'))

Mounted at /content/drive
app.py saved!
Location: /content/drive/MyDrive/FraudDetection/app.py


In [ ]:
!pip install flask flask-cors tensorflow pyngrok -q
print('All libraries installed!')

All libraries installed!


In [ ]:
import subprocess

result = subprocess.run(
    ['python', '-c',
     'import flask; import flask_cors; import tensorflow; print("All imports OK")'],
    capture_output=True, text=True, timeout=30
)
print('STDOUT:', result.stdout)
print('STDERR:', result.stderr[:200] if result.stderr else 'None')

STDOUT: All imports OK

STDERR: 2026-05-06 09:38:03.665825: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-06 09:38:03.776101: I tensorflow/core/platfo


In [ ]:
import threading
import time
import os
import subprocess

# Kill old processes
os.system('pkill -f app.py 2>/dev/null')
os.system('fuser -k 5000/tcp 2>/dev/null')
time.sleep(3)

# Start Flask and save logs
def run_flask():
    os.system(
        'python /content/drive/MyDrive/FraudDetection/app.py '
        '> /tmp/flask.log 2>&1'
    )

thread = threading.Thread(target=run_flask)
thread.daemon = True
thread.start()

print('Flask starting...')
print('Waiting up to 60 seconds for Neural Network to load...')
print()

# Check every 5 seconds
for i in range(12):
    time.sleep(5)
    elapsed = (i + 1) * 5

    # Check if Flask is responding
    check = subprocess.run(
        ['curl', '-s', '--max-time', '3', 'http://localhost:5000/'],
        capture_output=True, text=True
    )

    if check.stdout:
        print(f'Flask is running! ({elapsed} seconds)')
        print('Response:', check.stdout[:300])
        break
    else:
        # Show latest log
        log = subprocess.run(
            ['tail', '-2', '/tmp/flask.log'],
            capture_output=True, text=True
        )
        last_line = log.stdout.strip()[-100:] if log.stdout else 'starting...'
        print(f'  {elapsed}s — {last_line}')
else:
    print()
    print('Flask did not start in 60 seconds')
    print('Showing full log:')
    log = subprocess.run(
        ['cat', '/tmp/flask.log'],
        capture_output=True, text=True
    )
    print(log.stdout[-800:])

Flask starting...
Waiting up to 60 seconds for Neural Network to load...

  5s — instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Flask is running! (10 seconds)
Response: {"accuracy":0.9994,"f1_score":0.8241,"message":"AXD-Fraud Detection API","model":"Neural Network","precision":0.8119,"recall":0.8367,"status":"running","threshold":0.25000000000000006}



In [ ]:
from pyngrok import ngrok
import os
import time

# Kill old ngrok
os.system('pkill -f ngrok 2>/dev/null')
time.sleep(2)

# Read the ngrok token from Colab Secrets (key icon in the left sidebar).
# Add a secret named NGROK_AUTHTOKEN. Never paste the token into the notebook.
from google.colab import userdata
ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))

# Connect
public_url = ngrok.connect(5000)

# Clean URL
raw = str(public_url)
if '"' in raw:
    API_URL = raw.split('"')[1]
else:
    API_URL = raw.strip()

print('='*50)
print('API IS LIVE')
print('='*50)
print(f'URL: {API_URL}')
print()
print('Paste this in App.js line 4:')
print(f'const API = "{API_URL}";')

In [ ]:
import requests

headers = {'ngrok-skip-browser-warning': 'true'}

print('Testing all endpoints...')
print()

# Test home
try:
    r = requests.get(f'{API_URL}/', headers=headers, timeout=10)
    data = r.json()
    print('HOME endpoint:')
    print(f'  Status:    {data.get("status")}')
    print(f'  Model:     {data.get("model")}')
    print(f'  Threshold: {data.get("threshold")}')
    print(f'  F1:        {data.get("f1_score")}')
    print()
except Exception as e:
    print('HOME error:', e)

# Test predict
try:
    import numpy as np
    features = np.random.uniform(-2, 2, 30).tolist()
    r = requests.post(
        f'{API_URL}/predict',
        json={'features': features},
        headers=headers,
        timeout=15
    )
    result = r.json()
    print('PREDICT endpoint:')
    print(f'  Label:       {result.get("label")}')
    print(f'  Probability: {result.get("fraud_probability")}%')
    print(f'  Risk:        {result.get("risk_level")}')
    print(f'  Threshold:   {result.get("threshold_used")}')
    print()
except Exception as e:
    print('PREDICT error:', e)

# Test drift
try:
    r = requests.get(f'{API_URL}/drift/status',
                     headers=headers, timeout=10)
    print('DRIFT endpoint:', r.json())
    print()
except Exception as e:
    print('DRIFT error:', e)

# Test all models
try:
    r = requests.get(f'{API_URL}/all_models',
                     headers=headers, timeout=10)
    data = r.json()
    print('ALL MODELS endpoint:')
    for m in data.get('models', []):
        print(f'  {m["name"]:25} {m["status"]}')
    print(f'  Selected: {data.get("selected_model")}')
except Exception as e:
    print('ALL MODELS error:', e)

In [ ]:
 from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import os
import tensorflow as tf
import pickle

PROC_PATH = '/content/drive/MyDrive/FraudDetection/data/processed'
MOD_PATH  = '/content/drive/MyDrive/FraudDetection/models'

X_test = np.load(os.path.join(PROC_PATH, 'X_test.npy'))
y_test = np.load(os.path.join(PROC_PATH, 'y_test.npy'))

nn = tf.keras.models.load_model(
    os.path.join(MOD_PATH, 'neural_network.h5'))

with open(os.path.join(MOD_PATH, 'best_threshold.pkl'), 'rb') as f:
    threshold = pickle.load(f)

print(f'Threshold: {threshold}')
print()

fraud_indices = np.where(y_test == 1)[0]

print('PASTE THESE INTO YOUR DASHBOARD ONE AT A TIME')
print('='*60)

count = 0
for idx in fraud_indices:
    values = X_test[idx]
    prob   = float(nn.predict(
        values.reshape(1,-1), verbose=0).flatten()[0])

    if prob >= threshold:
        count += 1
        vals = ', '.join([str(round(v, 4)) for v in values.tolist()])
        print(f'\nFraud {count} — Model says {prob*100:.1f}% fraud:')
        print(vals)
        if count >= 3:
            break

print()
print('='*60)
print('Copy Fraud 1 values into dashboard and click Detect Fraud')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Threshold: 0.25000000000000006

PASTE THESE INTO YOUR DASHBOARD ONE AT A TIME

Fraud 1 — Model says 100.0% fraud:
-1.2712, 2.4627, -2.8514, 2.3245, -1.3722, -0.9482, -3.0652, 1.1669, -2.2688, -4.8811, 2.2551, -4.6864, 0.6524, -6.1743, 0.5944, -4.8497, -6.5365, -3.1191, 1.7155, 0.5605, 0.6529, 0.0819, -0.2213, -0.5236, 0.2242, 0.7563, 0.6328, 0.2502, -0.3532, -0.7961

Fraud 2 — Model says 100.0% fraud:
-13.1927, 12.786, -9.9067, 3.3203, -4.8012, 5.7601, -18.7509, -37.3534, -0.3915, -5.0525, 4.4068, -4.6108, -1.9095, -9.0727, -0.2261, -6.2116, -6.2481, -3.1492, 0.0516, -3.493, 27.2028, -8.887, 5.3036, -0.6394, 0.2632, -0.1089, 1.2696, 0.9394, -0.3492, -0.5603

Fraud 3 — Model says 100.0% fraud:
-2.8664, 2.3469, -4.0533, 3.9834, -3.4632, -1.281, -4.4748, 1.2167, -2.3098, -5.5155, 3.0484, -7.1283, 0.006, -7.4952, -0.2511, -7.568, -12.2236, -4.8952, 2.525, 0.27, 1.0497, 0.4758, 0.4045, 0.282, -0.5069, -0.3717, 0.6153, 0.8032, 0.1447, -0.405

Copy Fraud 1 values into dashboard and click Dete